# 05 Test set and final refit

Workflow steps 6-8. The model, its predictor subset and the threshold are frozen from notebooks 03 and 04 (`results/selected_model.json`, `results/operating_point.json`). This notebook loads the test set for the first and only time (793 wines, one row per wine), scores the frozen model on it, reports the test results side by side with the CV estimates, and then refits the same specification on all 3,961 wines. Nothing is changed after the test results are seen.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import (average_precision_score, roc_auc_score, roc_curve,
                             precision_recall_curve, confusion_matrix)

from wine_utils import (load_train, load_test, load_split, load_all, transform, build_selected,
                        expected_cost, ALL_FEATURES, C_FN_BASE)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

spec = json.load(open('results/selected_model.json'))
op = json.load(open('results/operating_point.json'))
assert op['model'] == spec['name']
t = op['threshold']
X_train, y_train = load_train()
print(f"Frozen: {spec['name']} on {spec['n_predictors']} predictors {spec['predictors']}, threshold {t:g}")

## Step 6: fit the frozen specification on the full training set

In [ ]:
model = build_selected(spec).fit(X_train, y_train)
assert model.named_steps['model'].n_features_in_ == len(spec['predictors'])
assert list(model[:-1].get_feature_names_out()) == spec['predictors']
print(spec['name'], 'fitted on', len(X_train), 'training wines')
model

## Step 7: score the test set, once

In [ ]:
X_test, y_test = load_test()
train_df, test_df = load_split('train'), load_split('test')
assert len(X_test) == 793 and not test_df.duplicated().any()
assert len(train_df.merge(test_df, how='inner')) == 0, 'a wine appears in both train and test'
print('Test wines:', len(y_test), '| share good:', round(y_test.mean(), 3),
      '| no duplicates, no overlap with training')

prob = model.predict_proba(X_test)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_test, prob >= t).ravel()
test = {'PR-AUC': average_precision_score(y_test, prob),
        'ROC-AUC': roc_auc_score(y_test, prob),
        f'Sensitivity @{t:g}': tp / (tp + fn),
        f'Specificity @{t:g}': tn / (tn + fp),
        f'Cost per wine @{t:g}': expected_cost(y_test, prob, t, C_FN_BASE)}

fig, ax = plt.subplots(figsize=(4, 3.5))
sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt='d', cbar=False, cmap='Blues', ax=ax,
            xticklabels=['not good', 'good'], yticklabels=['not good', 'good'])
ax.set(xlabel='Predicted', ylabel='Actual', title=f'Test set ({len(y_test)} wines), threshold {t:g}')
plt.tight_layout()
plt.show()

### CV estimates and test results side by side

CV values are mean ± SE over the 10 folds of the nested CV in notebook 03 (out-of-fold predictions of the selected model; sensitivity, specificity and cost at the chosen threshold are computed per fold from the same predictions).

In [ ]:
oof = pd.read_csv('results/oof_probabilities.csv', index_col=0)
cv_folds = []
for f, d in oof.groupby('fold'):
    p, yy = d[spec['name']].to_numpy(), d['y'].to_numpy()
    c_tn, c_fp, c_fn, c_tp = confusion_matrix(yy, p >= t, labels=[0, 1]).ravel()
    cv_folds.append({'PR-AUC': average_precision_score(yy, p), 'ROC-AUC': roc_auc_score(yy, p),
                     f'Sensitivity @{t:g}': c_tp / (c_tp + c_fn), f'Specificity @{t:g}': c_tn / (c_tn + c_fp),
                     f'Cost per wine @{t:g}': expected_cost(yy, p, t, C_FN_BASE)})
cv_folds = pd.DataFrame(cv_folds)
assert np.isclose(cv_folds['PR-AUC'].mean(), spec['cv_pr_auc_mean'])

comparison = pd.DataFrame({
    'CV mean': cv_folds.mean(),
    'CV SE': cv_folds.std() / np.sqrt(len(cv_folds)),
    'Test': pd.Series(test),
})
comparison['|Test - CV|'] = (comparison['Test'] - comparison['CV mean']).abs()
comparison.round(3)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, prob)
prec, rec, _ = precision_recall_curve(y_test, prob)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, lw=2, label=f"Test (AUC {result['ROC-AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], color='grey', ls='--', lw=0.8, label='No skill')
axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title=f'Test ROC curve, {spec["name"]}')
axes[1].plot(rec, prec, lw=2, label=f"Test (AP {result['PR-AUC']:.3f})")
axes[1].axhline(y_test.mean(), color='grey', ls='--', lw=0.8, label=f'No skill ({y_test.mean():.2f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title=f'Test precision-recall curve, {spec["name"]}')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 8: refit the same specification on all data (3,961 wines)

Only the parameters are re-estimated. The selected predictor subset, transformations, model family and threshold stay frozen, and the performance figures to report remain the step 7 test results. Predictors are standardised over the full data, so effects are per one-SD increase.

- **Logistic:** sklearn gives no standard errors, so the same unpenalised model is refitted in `statsmodels`. Each row is a different wine, so the usual model-based SEs are used. We report odds ratios per 1 SD.
- **LDA:** standardised discriminant coefficients (the weight of each predictor in the linear discriminant) and the standardised difference in class means.
- **Gaussian NB:** the standardised difference in class means for each predictor, which drives its per-feature likelihood ratio.

In [ ]:
df_all = load_all()
assert len(df_all) == 3961
X_all = transform(df_all[ALL_FEATURES])
y_all = (df_all['quality'] >= 7).astype(int)

final_model = build_selected(spec).fit(X_all, y_all)   # the deployable model; selects spec['predictors']
Z_all = X_all[spec['predictors']]
Z_all = (Z_all - Z_all.mean()) / Z_all.std()

if spec['name'] == 'Logistic':
    final = sm.Logit(y_all, sm.add_constant(Z_all)).fit(disp=0)
    print(final.summary())
    ci = final.conf_int()
    effects = pd.DataFrame({
        'coef (log-odds per SD)': final.params,
        'odds ratio per SD': np.exp(final.params),
        'OR 95% CI low': np.exp(ci[0]),
        'OR 95% CI high': np.exp(ci[1]),
        'p-value': final.pvalues,
    }).drop('const').sort_values('odds ratio per SD', ascending=False)
else:
    mean_diff = Z_all[y_all == 1].mean() - Z_all[y_all == 0].mean()
    effects = pd.DataFrame({'class mean difference (good - not good, SDs)': mean_diff})
    if spec['name'] == 'LDA':
        lda = final_model.named_steps['model']
        effects['standardised discriminant coefficient'] = lda.scalings_[:, 0]
    effects = effects.sort_values(effects.columns[-1], key=np.abs, ascending=False)
effects.round(3)

## Limitations

- **Binarised outcome.** Quality is cut at 7, so the ordering within each class (a 3 vs a 6, a 7 vs a 9) is lost, and wines just either side of the cut are treated as fully different.
- **Noisy labels.** Each quality score is the median of at least three expert ratings, so some labels near the cut-off are effectively uncertain, which caps achievable accuracy.
- **One region and vintage period.** The data are Vinho Verde wines from a 2009 study (Cortez et al.). Results may not transfer to other regions, styles or later vintages.
- **Missing drivers.** There is no price, grape variety, brand or cost information, so the cost ratio $c_{FN}/c_{FP}$ is assumed, not measured (see the sensitivity analysis in notebook 04).
- **White wine only.** No conclusion is drawn about red wine.
- **Simplicity has a price.** Each classifier uses the simplest predictor subset within one SE of its best (notebook 03), which gives up a little PR-AUC; the paired difference against all 11 predictors is reported in notebook 03. Correlated predictors (density, residual sugar, alcohol) can stand in for each other, so the odds ratios above should be read together rather than one at a time.
- **CV variance.** CV estimates come from one 10-fold split of 3,168 wines. The fold-to-fold SE (about 0.017 in PR-AUC) is larger than the gaps between the three classifiers, so the choice between them is not clear-cut, and the test set is one further draw of 793 wines.